# imporst

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
import gc
import time

## Загрузка данных

In [2]:
maker_trades = pd.read_parquet('./data/' + 'train/target_data_solusdt.parquet').reset_index()
answers = pd.read_parquet('./data/' + 'train/target_solusdt.parquet')
maker_trades['target'] = answers['target'].values
maker_trades['weight'] = answers['weight'].values
print(maker_trades.shape)
maker_trades.head()

(6156387, 6)


,timestamp_ns,side,price,amount,target,weight
0,1726617603492082000,1,131.466995,7.0,0,0.397396
1,1726617603512695000,1,131.466995,1.0,0,0.150202
2,1726617603512698000,1,131.466003,2.0,0,0.207696
3,1726617603543949000,-1,131.460007,8.0,1,0.433219
4,1726617603710144000,1,131.459000,1.0,0,0.120686


In [3]:
maker_trades.groupby((2 ** np.floor(np.log2(maker_trades.amount))).rename('min_log_ind')).agg(
    n=('target', 'size'),
    weight_mean=('weight', 'mean'),
    weight_sum=('weight', 'sum'),
    target_mean=('target', 'mean')
)

,n,weight_mean,weight_sum,target_mean
min_log_ind,,,,
1.0,1876248,0.154214,289344.392643,0.460629
2.0,1123367,0.236908,266135.034487,0.473797
4.0,790429,0.353024,279040.796685,0.474100
8.0,693295,0.518827,359700.089358,0.471329
16.0,617116,0.744849,459658.022505,0.475034
32.0,438629,1.004741,440708.361283,0.489986
64.0,242266,1.365298,330765.368684,0.572524
128.0,228572,1.801143,411690.802943,0.672278
256.0,117905,2.415053,284746.822870,0.728502


## Сделаем признаки из стакана сделок

In [4]:
eps = 1e-9

def last_less_than(times, t):
    return np.searchsorted(times, t, side='left') - 1
 
def read_book(parts, times, type_of_buy, prices):
    times_in_orderbook = np.concatenate([pq.ParquetFile(i).read_row_groups(j, columns=['timestamp_ns']).column(0).to_numpy()
                             for i, j in parts])
    times_indexes = last_less_than(times_in_orderbook, times)
    col_sames_price = np.zeros(len(times), np.float32)
    col_good_prices = np.zeros(len(times), np.float32)
    col_good_levels = np.zeros(len(times), np.float32)
    in_book = np.zeros(len(times), np.float32)
    best_opp = np.zeros(len(times), np.float32)
    cols = []
    for i in range(20):
        cols += [f'asks[{i}].price', f'asks[{i}].amount', f'bids[{i}].price', f'bids[{i}].amount']
    metrics = {}
    cur_iter = 0
    for path, rows in parts:
        pfile = pq.ParquetFile(path)
        for cur in rows:
            cur_window = pfile.read_row_group(cur, columns=cols)
            part_ord_lev = np.stack([cur_window.column(c).to_numpy() for c in cols], 1).reshape(-1, 20, 4)
            
            ap = np.round(part_ord_lev[:, :, 0].astype(np.float64) / 0.001)
            ac = part_ord_lev[:, :, 1].astype(np.float64)
            bp = np.round(part_ord_lev[:, :, 2].astype(np.float64) / 0.001)
            bc = part_ord_lev[:, :, 3].astype(np.float64)
            stats = {}
            stats['mid'] = (ap[:, 0] + bp[:, 0]) / 2
            stats['diff'] = (ap[:, 0] - bp[:, 0]).astype(np.float32)
            stats['a_opt'] = ap[:, 0]
            stats['b_opt'] = bp[:, 0]
            stats['cola_opt'] = ac[:, 0].astype(np.float32)
            stats['colb_opt'] = bc[:, 0].astype(np.float32)
            mid = (ap[:, 0] + bp[:, 0]) / 2
            stats['micro'] = ((ap[:, 0] * bc[:, 0] + bp[:, 0] * ac[:, 0]) / (bc[:, 0] + ac[:, 0]) - mid).astype(np.float32)
            a_sums_best = np.cumsum(ac, 1)
            b_sums_best = np.cumsum(bc, 1)
            for k in [1, 2, 3, 5, 10, 20]:
                stats[f'imb{k}'] = ((b_sums_best[:, k-1] - a_sums_best[:, k-1]) / (b_sums_best[:, k-1] + a_sums_best[:, k-1])).astype(np.float32)
            stats['lda5'] = np.log1p(a_sums_best[:, 4]).astype(np.float32)
            stats['ldb5'] = np.log1p(b_sums_best[:, 4]).astype(np.float32)
            stats['lda20'] = np.log1p(a_sums_best[:, 19]).astype(np.float32)
            stats['ldb20'] = np.log1p(b_sums_best[:, 19]).astype(np.float32)
            for w in [3, 10]:
                a_window = (ac * ((ap - mid[:, None]) <= w)).sum(1)
                b_window = (bc * ((mid[:, None] - bp) <= w)).sum(1)
                stats[f'imbw{w}'] = ((b_window - a_window) / (b_window + a_window + eps)).astype(np.float32)
                stats[f'ldaw{w}'] = np.log1p(a_window).astype(np.float32)
                stats[f'ldbw{w}'] = np.log1p(b_window).astype(np.float32)   
            stats['amean_df_mid'] = ((ac[:, :10] * (ap[:, :10] - mid[:, None])).sum(1) / (a_sums_best[:, 9])).astype(np.float32)    
            stats['bmean_df_mid'] = ((bc[:, :10] * (mid[:, None] - bp[:, :10])).sum(1) / (b_sums_best[:, 9])).astype(np.float32)
            
            for i, j in stats.items():
                metrics.setdefault(i, []).append(j)
 
            ind_inbook = np.where((times_indexes >= cur_iter) & (times_indexes < cur_iter + len(part_ord_lev)))[0]
            real = times_indexes[ind_inbook] - cur_iter
            types = type_of_buy[ind_inbook]
            price = prices[ind_inbook][:, None]
            cost = np.where(types[:, None] == 1, bp[real], ap[real])
            amount = np.where(types[:, None] == 1, bc[real], ac[real])
            col_sames_price[ind_inbook] = (amount * (cost == price)).sum(1)
            col_good_prices[ind_inbook] = (amount * np.where(types[:, None] == 1, cost > price, cost < price)).sum(1)
            col_good_levels[ind_inbook] = np.where(types[:, None] == 1, cost > price, cost < price).sum(1)
            in_book[ind_inbook] = (cost == price).any(1)
            best_opp[ind_inbook] = np.where(types == 1, ac[real, 0], bc[real, 0])
            cur_iter += len(part_ord_lev)
    book = {i: np.concatenate(j) for i, j in metrics.items()}
    book['time'] = times_in_orderbook
    temp = dict(col_sames_price=col_sames_price, col_good_prices=col_good_prices, col_good_levels=col_good_levels, in_book=in_book, best_opp=best_opp)
    return book, times_indexes, temp

In [5]:
def prepare_trade(parts):
    temp = pd.concat([pq.ParquetFile(i).read_row_groups(j).to_pandas().reset_index() for i, j in parts],
                   ignore_index=True)
    t = {}
    for inst in ['SOLUSDT', 'BTCUSDT', 'ETHUSDT']:
        a = temp[temp.instrument == inst].sort_values('timestamp_ns')
        time_ends = {'time': a.timestamp_ns.values}
        time_ends['pref_bam'] = np.concatenate([[0], np.cumsum(a.bid_amount.fillna(0).values)])
        time_ends['pref_askam'] = np.concatenate([[0], np.cumsum(a.ask_amount.fillna(0).values)])
        time_ends['pref_bidc'] = np.concatenate([[0], np.cumsum(a.bid_count.fillna(0).values)])
        time_ends['pref_askc'] = np.concatenate([[0], np.cumsum(a.ask_count.fillna(0).values)])
        if inst == 'SOLUSDT':
            hi = np.fmax(a.bid_max_price.values, a.ask_max_price.values).astype(np.float64) / 0.001
            lo = np.fmin(a.bid_min_price.values, a.ask_min_price.values).astype(np.float64) / 0.001
            for n in [4, 20, 120, 240]:
                time_ends[f'hi{n}'] = pd.Series(hi).rolling(n, min_periods=1).max().values
                time_ends[f'lo{n}'] = pd.Series(lo).rolling(n, min_periods=1).min().values
        t[inst] = time_ends
    return t

def load_btc_eth_books(parts):
    temp = pd.concat([pq.ParquetFile(i).read_row_groups(j).to_pandas().reset_index() for i, j in parts],
                   ignore_index=True)
    t = {}
    for inst in ['BTCUSDT', 'ETHUSDT']:
        a = temp[temp.instrument == inst].sort_values('timestamp_ns', kind='stable')
        t[inst] = {'time': a.timestamp_ns.values, 'mid': a.mid_price.values.astype(np.float64)}
        for v in ['vwap_1_lvl', 'vwap_3_lvl', 'vwap_5_lvl', 'vwap_10_lvl']:
            t[inst][v] = a[v].values.astype(np.float32)
    return t

def rng_parts(path):
    return list(range(pq.ParquetFile(path).num_row_groups))

In [6]:
def build_features(data, book, times_indexes, f1, pref_sums, btc_eth_books, hist):
    def to32():
        for i in features:
            if features[i].dtype != np.float32:
                features[i] = np.asarray(features[i], dtype=np.float32)
    
    times = data.timestamp_ns.values
    types = data.side.values.astype(np.float64)
    prices = np.round(data.price.values.astype(np.float64) / 0.001)
    amounts = data.amount.values.astype(np.float64)
    features = {}
    features['types'] = types
    features['lamt'] = np.log(amounts)

    features['or_delta'] = types * (book['mid'][times_indexes] - prices)
    features['cush_micro'] = types * (book['mid'][times_indexes] + book['micro'][times_indexes] - prices)
    features['thr'] = 3e-5 * prices
    features['margin'] = 3e-5 * prices + types * (book['mid'][times_indexes] - prices)
    features['dmargin'] = (np.ceil(2 * 3e-5 * prices - 1e-9) - 1) / 2 + types * (book['mid'][times_indexes] - prices)
    features['diff'] = book['diff'][times_indexes]
    features['micro'] = types * book['micro'][times_indexes]
    for i in [1, 2, 3, 5, 10, 20]:
        features[f'imb{i}'] = types * book[f'imb{i}'][times_indexes]
    features['imbw3'] = types * book['imbw3'][times_indexes]
    features['imbw10'] = types * book['imbw10'][times_indexes]
    
    for a, b, nm in [('ldb5', 'lda5', 'd5'), ('ldb20', 'lda20', 'd20'), ('ldbw3', 'ldaw3', 'dw3'),
                     ('ldbw10', 'ldaw10', 'dw10'), ('bmean_df_mid', 'amean_df_mid', 'wd10')]:
        features['own_' + nm] = np.where(types == 1, book[a][times_indexes], book[b][times_indexes])
        features['opp_' + nm] = np.where(types == 1, book[b][times_indexes], book[a][times_indexes])
    
    features['col_sames_price'] = np.log1p(f1['col_sames_price'])
    features['col_good_prices'] = np.log1p(f1['col_good_prices'])
    features['col_good_levels'] = f1['col_good_levels']
    features['in_book'] = f1['in_book']
    features['best_opp'] = np.log1p(f1['best_opp'])
    features['amt_lvl'] = amounts / (f1['col_sames_price'] + 1)
    features['amt_through'] = amounts / (f1['col_sames_price'] + f1['col_good_prices'] + 1)
    features['was_older'] = (times - book['time'][times_indexes]) / 1e6
    
    mid_notsm = np.concatenate([[True], np.diff(book['mid']) != 0])
    last_chg = np.maximum.accumulate(np.where(mid_notsm, np.arange(len(mid_notsm)), 0))
    features['dt_midchg'] = (times - book['time'][last_chg[times_indexes]]) / 1e6
    to32()

    cabs = np.concatenate([[0], np.cumsum(np.abs(np.diff(book['mid'])))])
    for i in [0.1, 0.25, 0.5, 1, 2, 3, 5, 10, 30, 60, 180]:
        j = np.maximum(last_less_than(book['time'], times - int(i * 10**9)), 0)
        features[f'ret{i}'] = types * (book['mid'][times_indexes] - book['mid'][j])
        if i in [1, 3, 10, 30, 180]:
            features[f'vol{i}'] = cabs[times_indexes] - cabs[j]
        if i in [0.5, 2, 10]:
            features[f'imb1_l{i}'] = types * book['imb1'][j]
            features[f'imb5_l{i}'] = types * book['imb5'][j]
    to32()

    t__ = np.zeros(len(book['b_opt']))
    t__[1:] = ((book['b_opt'][1:] >= book['b_opt'][:-1]) * book['colb_opt'].astype(np.float64)[1:] - (book['b_opt'][1:] <= book['b_opt'][:-1]) * book['colb_opt'].astype(np.float64)[:-1]
             - (book['a_opt'][1:] <= book['a_opt'][:-1]) * book['cola_opt'].astype(np.float64)[1:] + (book['a_opt'][1:] >= book['a_opt'][:-1]) * book['cola_opt'].astype(np.float64)[:-1])
    t1 = np.concatenate([[0], np.cumsum(t__)])
    t2 = np.concatenate([[0], np.cumsum(np.abs(t__))])
    for i in [0.25, 1, 3, 10, 30]:
        j = np.maximum(last_less_than(book['time'], times - int(i * 10**9)), 0)
        features[f'ofi{i}'] = types * (t1[times_indexes + 1] - t1[j + 1]) / (t2[times_indexes + 1] - t2[j + 1] + 1)
        if i in [1, 10]:
            features[f'ofir{i}'] = types * (t1[times_indexes + 1] - t1[j + 1])
    features['mz30'] = features['dmargin'] / (features['vol30'] / 10 + 1)
    features['mz180'] = features['dmargin'] / (features['vol180'] / 60 + 1)
    features['cz30'] = features['or_delta'] / (features['vol30'] / 10 + 1)
    to32()

    for inst, ch in [('SOLUSDT', 's'), ('BTCUSDT', 'b'), ('ETHUSDT', 'e')]:
        was = last_less_than(pref_sums[inst]['time'], times) + 1
        if ch == 's':
            features['dt_agg'] = (times - pref_sums[inst]['time'][was - 1]) / 1e6
        for n in [1, 2, 4, 8, 20, 40, 120, 240]:
            if ch != 's' and n == 240:
                continue
            was0 = np.maximum(was - n, 0)
            bgood = pref_sums[inst]['pref_bam'][was] - pref_sums[inst]['pref_bam'][was0]
            sentgood = pref_sums[inst]['pref_askam'][was] - pref_sums[inst]['pref_askam'][was0]
            features[f'{ch}flow{n}'] = -types * (bgood - sentgood) / (bgood + sentgood + eps)
            if n in [1, 4, 20, 120]:
                features[f'{ch}lvol{n}'] = np.log1p(bgood + sentgood)
            if ch == 's' and n in [1, 4, 20]:
                bc = pref_sums[inst]['pref_bidc'][was] - pref_sums[inst]['pref_bidc'][was0]
                sc = pref_sums[inst]['pref_askc'][was] - pref_sums[inst]['pref_askc'][was0]
                features[f'scnt{n}'] = -types * (bc - sc) / (bc + sc + 1e-9)
        if ch == 's':
            for n in [4, 20, 120, 240]:
                hi = pref_sums[inst][f'hi{n}'][was - 1]
                lo = pref_sums[inst][f'lo{n}'][was - 1]
                features[f'sol_range{n}'] = hi - lo
                features[f'mid_vs_range{n}'] = types * (book['mid'][times_indexes] - (hi + lo) / 2)
                features[f'price_vs_range{n}'] = types * ((hi + lo) / 2 - prices)
            features['mzr20'] = features['dmargin'] / (features['sol_range20'] + 1)
        to32()

    for inst, ch in [('BTCUSDT', 'b'), ('ETHUSDT', 'e')]:
        t3 = last_less_than(btc_eth_books[inst]['time'], times)
        features[f'{ch}_dt'] = (times - btc_eth_books[inst]['time'][t3]) / 1e6
        for d in [0.1, 0.25, 0.5, 1, 2, 5, 10, 30, 60]:
            last_d = np.maximum(last_less_than(btc_eth_books[inst]['time'], times - int(d * 10**9)), 0)
            plus_proc = types * (btc_eth_books[inst]['mid'][t3] / btc_eth_books[inst]['mid'][last_d] - 1) * 1e4
            features[f'{ch}plus_proc{d}'] = plus_proc
            features[f'{ch}res{d}'] = plus_proc - features[f'ret{d}'] / prices * 1e4
        for v, short in [('vwap_1_lvl', 'vw1'), ('vwap_3_lvl', 'vw3'), ('vwap_5_lvl', 'vw5'), ('vwap_10_lvl', 'vw10')]:
            features[f'{ch}_{short}'] = types * (btc_eth_books[inst][v][t3] / btc_eth_books[inst]['mid'][t3] - 1) * 1e4
        to32()

    last_pay = last_less_than(hist.ts.values, times)
    t5 = np.maximum(last_pay, 0)
    mask = last_pay >= 0
    features['h_dt'] = np.where(mask, (times - hist.ts.values[t5]) / 1e6, 1e7)
    features['h_side'] = np.where(mask, types * hist.side.values.astype(np.float64)[t5], 0)
    features['h_px'] = np.where(mask, types * (hist.prices.values[t5] - prices), np.nan)
    features['h_pxmid'] = np.where(mask, types * (hist.prices.values[t5] - book['mid'][times_indexes]), np.nan)
    csa = np.concatenate([[0], np.cumsum(hist.amount.values.astype(np.float64))])
    css = np.concatenate([[0], np.cumsum(hist.side.values.astype(np.float64) * hist.amount.values.astype(np.float64))])
    csp = np.concatenate([[0], np.cumsum(hist.prices.values * hist.amount.values.astype(np.float64))])
    for w in [0.001, 0.01, 0.05, 0.25, 1, 5]:
        h0 = last_less_than(hist.ts.values, times - int(w * 10**9))
        t1 = csa[last_pay + 1] - csa[h0 + 1]
        t2 = css[last_pay + 1] - css[h0 + 1]
        features[f'h_cnt{w}'] = last_pay - h0
        features[f'h_flow{w}'] = types * t2 / (t1 + 1e-9)
        if w in [0.05, 1]:
            features[f'h_lam{w}'] = np.log1p(t1)
        if w in [0.25, 5]:
            sp = csp[last_pay + 1] - csp[h0 + 1]
            features[f'h_vwap{w}'] = np.where(t1 > 0, types * (sp / (t1 + 1e-9) - prices), np.nan)
    to32()

    n_aft = np.zeros(len(times)); sv_aft = np.zeros(len(times))
    same_vol = np.zeros(len(times)); same_cnt = np.zeros(len(times))
    best_aft = np.full(len(times), np.nan); worst_aft = np.full(len(times), np.nan)
    sw_cnt = np.zeros(len(times)); sw_vol = np.zeros(len(times)); sw_span = np.zeros(len(times))
    for k in range(30):
        idx = last_pay - k
        valid = idx >= 0
        idx = np.maximum(idx, 0)
        tk, sk, pk, ak = hist.ts.values[idx], hist.side.values.astype(np.float64)[idx], hist.prices.values[idx], hist.amount.values.astype(np.float64)[idx]
        rel = types * (pk - prices)
        if k < 3:
            features[f'h{k}_dt'] = np.where(valid, (times - tk) / 1e6, 1e7)
            features[f'h{k}_side'] = np.where(valid, types * sk, 0)
            features[f'h{k}_px'] = np.where(valid, rel, np.nan)
            features[f'h{k}_am'] = np.where(valid, np.log(ak), np.nan)
        aft = valid & (tk > book['time'][times_indexes])
        n_aft += aft
        sv_aft += aft * sk * ak
        same = aft & (pk == prices) & (sk == types)
        same_vol += same * ak
        same_cnt += same
        best_aft = np.where(aft, np.fmax(best_aft, rel), best_aft)
        worst_aft = np.where(aft, np.fmin(worst_aft, rel), worst_aft)
        sw = valid & ((times - tk) < 2_000_000) & (sk == types)
        sw_cnt += sw
        sw_vol += sw * ak
        sw_span = np.where(sw, np.fmax(sw_span, rel), sw_span)
    features['aft_cnt'] = n_aft
    features['aft_flow'] = types * sv_aft
    features['aft_same_vol'] = same_vol
    features['aft_same_cnt'] = same_cnt
    features['lvl_left'] = f1['col_sames_price'] - 3 * same_vol
    features['aft_best'] = best_aft
    features['aft_worst'] = worst_aft
    features['sw_cnt'] = sw_cnt
    features['sw_vol'] = np.log1p(sw_vol)
    features['sw_span'] = sw_span
    to32()

    sig = features['vol30'] / 10 + 0.5
    sig_d = sig * 1e4 / prices
    for d in [1, 3, 10, 30, 180]:
        features[f'plus_proc{d}'] = features[f'ret{d}'] / (features[f'vol{d}'] + 1)
    for d in [0.5, 1, 3, 10]:
        features[f'retz{d}'] = features[f'ret{d}'] / sig
    for inst, nm in [('BTCUSDT', 'b'), ('ETHUSDT', 'e')]:
        t6 = last_less_than(btc_eth_books[inst]['time'], times)
        cab = np.concatenate([[0], np.cumsum(np.abs(np.diff(np.log(btc_eth_books[inst]['mid'])))) * 1e4])
        for d in [10, 60]:
            je = np.maximum(last_less_than(btc_eth_books[inst]['time'], times - d * 10**9), 0)
            v = cab[t6] - cab[je]
            features[f'{nm}vol{d}'] = v
            features[f'{nm}eff{d}'] = features[f'{nm}plus_proc{d}'] / (v + 0.1)
        for d in [0.5, 1, 2, 5]:
            features[f'{nm}resz{d}'] = features[f'{nm}res{d}'] / sig_d
        features[f'{nm}volr'] = features[f'{nm}vol60'] / (features['vol30'] * 2 * 1e4 / prices + 0.1)
    features['volr'] = features['vol10'] / (features['vol180'] / 18 + 1)
    features['amt_rel'] = features['lamt'] - features['slvol20']
    features['margz'] = features['dmargin'] / sig

    features['hour'] = (times // (3600 * 10**9)) % 24
    to32()
    return pd.DataFrame(features)

P.S: константа 3e-5 взялась не из будущего: она взялась из рассмотра, как на train как соотносятся mid и target. Если надо - я могу дописать код, которым я её получил (но сейчас не буду, т.к. различные тестирования я проводил в другом notebook, а там всё +- удалено, но если надо - допишу).

## Загрузка train

In [7]:
BOOK_TR = './data/' + 'train/orderbook_solusdt.parquet'
AGG_TR = './data/' + 'train/agg_trades.parquet'
EMB_TR = './data/' + 'train/orderbook_embedding.parquet'

norm_price = np.round(maker_trades.price.values.astype(np.float64) / 0.001)
book, times_indexes, book_stat = read_book([(BOOK_TR, rng_parts(BOOK_TR))], maker_trades.timestamp_ns.values, maker_trades.side.values, norm_price)
agg = prepare_trade([(AGG_TR, rng_parts(AGG_TR))])
emb = load_btc_eth_books([(EMB_TR, rng_parts(EMB_TR))])
print(len(book['time']))

16083804


## train и валидация

In [8]:
def make_train_features(seed):
    keep = np.random.default_rng(seed).random(len(maker_trades)) < 1 / 3
    hist = pd.DataFrame({'ts': maker_trades.timestamp_ns.values[keep], 'side': maker_trades.side.values[keep],
                         'prices': norm_price[keep], 'amount': maker_trades.amount.values[keep]})
    T = build_features(maker_trades, book, times_indexes, book_stat, agg, emb, hist)
    return T[mask].reset_index(drop=True)

mask = maker_trades.timestamp_ns.values >= maker_trades.timestamp_ns.values[0] + 300 * 10**9
ts_tr = maker_trades.timestamp_ns.values[mask]
y_tr = maker_trades.target.values[mask]
w_tr = maker_trades.weight.values[mask]

TR_F = make_train_features(42)
FEATS = list(TR_F.columns)

In [9]:
def wlogloss(y, p, w):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.sum(w * (y * np.log(p) + (1 - y) * np.log(1 - p))) / np.sum(w)

PARAMS = {
    'extra': dict(objective='binary', learning_rate=0.05, bagging_fraction=0.7, bagging_freq=1,
                   max_bin=255, verbose=-1, extra_trees=True, num_leaves=255, min_data_in_leaf=3000, feature_fraction=0.5, lambda_l2=50),
    'reg': dict(objective='binary', learning_rate=0.05, bagging_fraction=0.7, bagging_freq=1,
                   max_bin=255, verbose=-1, num_leaves=63, min_data_in_leaf=10000, feature_fraction=0.4, lambda_l2=100),
}

full = lgb.Dataset(TR_F, y_tr, weight=w_tr, free_raw_data=False)
full.construct()

day_tr = pd.to_datetime(ts_tr).day.values
best_its = {'extra': [], 'reg': []}
scores = []

for tr_days, va_days in [(list(range(19, 29)), [18]),
         (list(range(18, 23)), [23, 24]),
         (list(range(18, 26)), [26, 27, 28])]:
    t0 = time.time()
    train_row = np.isin(day_tr, tr_days)
    val_raw = np.isin(day_tr, va_days)
    preds = {}
    for name, prm in PARAMS.items():
        model_val = lgb.train(prm, full.subset(np.where(train_row)[0]), 3000, valid_sets=[full.subset(np.where(val_raw)[0])], callbacks=[lgb.early_stopping(100, verbose=False)])
        preds[name] = model_val.predict(TR_F[val_raw], num_iteration=model_val.best_iteration)
        best_its[name].append(model_val.best_iteration)
    sc = wlogloss(y_tr[val_raw], (preds['extra'] + preds['reg']) / 2, w_tr[val_raw])
    scores.append(sc)
    print(f'валидация {va_days}:  extra {wlogloss(y_tr[val_raw], preds["extra"], w_tr[val_raw]):.5f}'
          f'  reg {wlogloss(y_tr[val_raw], preds["reg"], w_tr[val_raw]):.5f}  среднее {sc:.5f}')

print('средний loss по фолдам:', round(np.mean(scores), 5))
BEST_IT = {i: int(np.mean(j)) for i, j in best_its.items()}
print('число итераций:', BEST_IT)

C:\Users\alex_s\anaconda3\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")
C:\Users\alex_s\anaconda3\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


валидация [18]:  extra 0.63560  reg 0.63465  среднее 0.63434


C:\Users\alex_s\anaconda3\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")
C:\Users\alex_s\anaconda3\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


валидация [23, 24]:  extra 0.64465  reg 0.64491  среднее 0.64384


C:\Users\alex_s\anaconda3\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")
C:\Users\alex_s\anaconda3\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


валидация [26, 27, 28]:  extra 0.61809  reg 0.61858  среднее 0.61704
средний loss по фолдам: 0.63174
число итераций: {'extra': 154, 'reg': 153}


валидация [23, 24]:  extra 0.64449  reg 0.64547  среднее 0.64410  константа 0.69483  (128 с)


валидация [26, 27, 28]:  extra 0.61819  reg 0.61864  среднее 0.61719  константа 0.69164  (343 с)


средний loss по фолдам: 0.63214
число итераций: {'extra': 142, 'reg': 142}


## Финал

In [10]:
SEEDS = [42, 52, 67, 69]
N_IT = {i: int(j * 1.1) for i, j in BEST_IT.items()}

models = []
for s in SEEDS:
    t0 = time.time()
    if s != 42:
        X = make_train_features(s)
        full = lgb.Dataset(X, y_tr, weight=w_tr)
        full.construct()
    for name, prm in PARAMS.items():
        models.append(lgb.train(dict(prm, seed=s), full, N_IT[name]))
    print('seed', s, 'DONE!', round(time.time() - t0), 'с')

seed 42 DONE! 347 с
seed 52 DONE! 551 с
seed 67 DONE! 554 с
seed 69 DONE! 526 с


seed 2024 готово за 266 с


43

## Test

In [11]:
test = pd.read_parquet('./data/' + 'test/target_data_solusdt.parquet').reset_index()
BOOK_TE = './data/' + 'test/orderbook_solusdt.parquet'
AGG_TE = './data/' + 'test/agg_trades.parquet'
EMB_TE = './data/' + 'test/orderbook_embedding.parquet'

norm_price = np.round(test.price.values.astype(np.float64) / 0.001)
def last(p):
    pf = pq.ParquetFile(p)
    return [i for i in range(pf.num_row_groups) if pf.metadata.row_group(i).num_rows > 0][-1:]

book_te, times_ind_te, f_te = read_book([(BOOK_TR, last(BOOK_TR)), (BOOK_TE, rng_parts(BOOK_TE))],
                                    test.timestamp_ns.values, test.side.values, norm_price)
agg_te = prepare_trade([(AGG_TR, last(AGG_TR)), (AGG_TE, rng_parts(AGG_TE))])
emb_te = load_btc_eth_books([(EMB_TR, last(EMB_TR)), (EMB_TE, rng_parts(EMB_TE))])

train_tail = maker_trades[maker_trades.timestamp_ns > maker_trades.timestamp_ns.max() - 600 * 10**9]
we_can_see = np.random.default_rng(0).random(len(train_tail)) < 1 / 3
hist_te = pd.DataFrame({'ts': np.concatenate([train_tail.timestamp_ns.values[we_can_see], test.timestamp_ns.values]),
                        'side': np.concatenate([train_tail.side.values[we_can_see], test.side.values]),
                        'prices': np.concatenate([np.round(train_tail.price.values[we_can_see].astype(np.float64) / 0.001), norm_price]),
                        'amount': np.concatenate([train_tail.amount.values[we_can_see], test.amount.values])})

X_te = build_features(test, book_te, times_ind_te, f_te, agg_te, emb_te, hist_te)
print(X_te.shape)

(711689, 238)


история до test, минут: стакан 99  agg 562  btc 203


(711689, 238)


In [12]:
pred = np.mean([m.predict(X_te[FEATS]) for m in models], axis=0)
pred = np.clip(pred, 1e-4, 1 - 1e-4)

sub = pd.DataFrame({'id': test['id'].values, 'prediction': pred})
sub.to_csv('submission.csv', index=False, float_format='%.6f')
print(sub.shape, sub.prediction.describe())
sub.head()

(711689, 2) count    711689.000000
mean          0.465434
std           0.143732
min           0.081395
25%           0.363979
50%           0.449115
75%           0.548579
max           0.982394
Name: prediction, dtype: float64


,id,prediction
0,0,0.139348
1,1,0.163110
2,2,0.181511
3,3,0.239054
4,4,0.247780
